### This is where we chain every step

In [1]:
import sys
from pathlib import Path
import yaml
import pandas as pd
project_dir = Path.cwd().parent
sys.path.append(str(project_dir))
config_path = project_dir / "config.yaml"

from src.corrupt import random_missing_values
from src.clean import mean_fill
from src.evaluate import evaluate_continuous_recovery
from src.load import load_model_data
from src.run_experiment import run_experiment

with config_path.open() as config_file:
    config = yaml.safe_load(config_file)

corruption_rate = config["corruption_rate"]
target_drug = config["target_drug"]
columns = config["columns"]
seed = config["seed"]

processed_dir = project_dir / "data/processed/"
drug_consumption = project_dir / f"data/processed/Drug_Consumption_{target_drug}.csv"

# load the dfs
original_df = load_model_data(drug_consumption)

# corrupt the df
corrupted_df, mask_df = random_missing_values(original_df, corruption_rate, seed)

# clean the df 
cleaned_df = mean_fill(corrupted_df, columns, {})

# evaluate the cleaned df
evaluated_metrics = evaluate_continuous_recovery(original_df, cleaned_df, mask_df, columns)

# print("Continuous Recovery Metrics")
# print("---------------------------")
# print(f"RMSE:             {evaluated_metrics['rmse']:.4f}")
# print(f"MAE:              {evaluated_metrics['mae']:.4f}")
# print(f"Bias:             {evaluated_metrics['bias']:.4f}")
# print(f"Variance Ratio:   {evaluated_metrics['variance_ratio']:.4f}")
# print(f"Wasserstein:      {evaluated_metrics['wasserstein']:.4f}")

result_df = run_experiment(original_df)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

parquet = pd.read_parquet(project_dir / "results/experiments.parquet")

parquet

,seed,corruption_rate,strategy,corruption_type,rmse,mae,bias,variance_ratio,wasserstein
0,1,0.1,mean_fill,random_missing_values,0.994414,0.799872,0.046914,0.000053,0.793510
1,1,0.1,mean_fill,mnar_missing_values,1.006205,0.797980,0.007642,0.000094,0.790182
2,1,0.2,mean_fill,random_missing_values,0.987827,0.795284,0.017840,0.000083,0.787552
3,1,0.2,mean_fill,mnar_missing_values,0.981888,0.781077,0.019973,0.000160,0.771735
4,1,0.3,mean_fill,random_missing_values,0.989110,0.792072,0.003456,0.000182,0.781359
5,1,0.3,mean_fill,mnar_missing_values,0.979808,0.781108,-0.003549,0.000192,0.771386
6,2,0.1,mean_fill,random_missing_values,0.969553,0.782628,0.009689,0.000048,0.776993
7,2,0.1,mean_fill,mnar_missing_values,0.992455,0.801170,0.002529,0.000070,0.793204
8,2,0.2,mean_fill,random_missing_values,0.981618,0.790360,0.031723,0.000150,0.782514
9,2,0.2,mean_fill,mnar_missing_values,1.000001,0.801915,-0.011226,0.000025,0.798210
